# Optimizers & Learning Rate Schedulers Visualized

**What you'll learn in this notebook:**
- Visualizing a 2D loss surface to build intuition
- Implementing SGD from scratch — the core update rule
- How momentum accelerates SGD (with trajectory plots)
- Adam step by step — running means, variance, bias correction
- AdamW vs Adam — why decoupled weight decay matters
- Side-by-side optimizer comparison on the same problem
- Learning rate schedulers visualized: StepLR, CosineAnnealing, OneCycleLR, warmup+cosine
- Parameter groups — different learning rates for different layers

**Prerequisites:** Notebooks 01-04 (Tensors through Training).  
**Time:** ~45 minutes  
**Hardware:** CPU only — no GPU required!

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import cm

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

## 1. Visualize a 2D Loss Surface

To build intuition, let's work with a loss function we can plot:

`f(x, y) = (x - 1)^2 + 10 * (y - 2)^2`

This is an elliptical bowl — steeper in the y-direction than x. This asymmetry makes optimization interesting.

In [ ]:
def loss_fn(x, y):
    """Elliptical bowl: minimum at (1, 2)."""
    return (x - 1)**2 + 10 * (y - 2)**2

# Create contour plot
xg = np.linspace(-4, 6, 200)
yg = np.linspace(-2, 6, 200)
X, Y = np.meshgrid(xg, yg)
Z = (X - 1)**2 + 10 * (Y - 2)**2

fig, ax = plt.subplots(figsize=(8, 6))
contour = ax.contour(X, Y, Z, levels=np.logspace(-0.5, 3, 20), cmap="viridis")
ax.clabel(contour, inline=True, fontsize=8)
ax.plot(1, 2, "r*", markersize=15, label="Minimum (1, 2)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Loss Surface: f(x,y) = (x-1)² + 10(y-2)²")
ax.legend()
plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/loss_surface.png", dpi=100, bbox_inches="tight")
plt.show()
print("Notice the elliptical contours — gradient descent will zigzag here.")

## 2. SGD from Scratch

Stochastic Gradient Descent is the simplest optimizer. The update rule:

`w = w - lr * gradient`

Let's implement it by hand and watch it navigate the loss surface.

In [ ]:
def optimize_trajectory(optimizer_fn, lr, steps=50, start=(-3.0, 5.0)):
    """Run an optimizer and record the trajectory."""
    x = torch.tensor([start[0]], requires_grad=True)
    y = torch.tensor([start[1]], requires_grad=True)
    opt = optimizer_fn([x, y], lr=lr)

    trajectory = [(x.item(), y.item())]
    for _ in range(steps):
        opt.zero_grad()
        loss = loss_fn(x, y)
        loss.backward()
        opt.step()
        trajectory.append((x.item(), y.item()))
    return trajectory

# SGD from scratch (manual implementation)
x = torch.tensor([-3.0], requires_grad=True)
y = torch.tensor([5.0], requires_grad=True)
lr = 0.05
trajectory_manual = [(x.item(), y.item())]

for step in range(50):
    loss = loss_fn(x, y)
    loss.backward()
    with torch.no_grad():
        x -= lr * x.grad
        y -= lr * y.grad
    x.grad.zero_()
    y.grad.zero_()
    trajectory_manual.append((x.item(), y.item()))

    if step < 5 or step % 10 == 9:
        print(f"Step {step:3d}: x={x.item():7.4f}, y={y.item():7.4f}, loss={loss.item():8.4f}")

print(f"\nFinal: ({x.item():.4f}, {y.item():.4f}), Target: (1.0, 2.0)")

## 3. SGD with Momentum

Momentum adds a "velocity" term that accumulates past gradients. This:
- Accelerates through consistent gradient directions
- Dampens oscillations in steep dimensions

Update rules:
```
v = momentum * v + gradient
w = w - lr * v
```

In [ ]:
traj_sgd = optimize_trajectory(
    lambda params, lr: torch.optim.SGD(params, lr=lr),
    lr=0.05, steps=50
)
traj_momentum = optimize_trajectory(
    lambda params, lr: torch.optim.SGD(params, lr=lr, momentum=0.9),
    lr=0.05, steps=50
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, traj, title in [(ax1, traj_sgd, "SGD (no momentum)"),
                          (ax2, traj_momentum, "SGD + Momentum (0.9)")]:
    ax.contour(X, Y, Z, levels=np.logspace(-0.5, 3, 20), cmap="viridis", alpha=0.5)
    xs, ys = zip(*traj)
    ax.plot(xs, ys, "r.-", linewidth=1, markersize=3)
    ax.plot(xs[0], ys[0], "go", markersize=8, label="Start")
    ax.plot(1, 2, "r*", markersize=15, label="Minimum")
    ax.set_title(title)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend()

plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/sgd_vs_momentum.png", dpi=100, bbox_inches="tight")
plt.show()
print("Momentum reduces zigzagging and converges faster!")

## 4. Adam Step by Step

Adam combines momentum (running mean of gradients) with RMSprop (running mean of squared gradients):

```
m = beta1 * m + (1-beta1) * g           # first moment (mean)
v = beta2 * v + (1-beta2) * g^2         # second moment (variance)
m_hat = m / (1 - beta1^t)               # bias correction
v_hat = v / (1 - beta2^t)               # bias correction
w = w - lr * m_hat / (sqrt(v_hat) + eps)
```

Let's implement this manually:

In [ ]:
x = torch.tensor([-3.0], requires_grad=True)
y = torch.tensor([5.0], requires_grad=True)

lr = 0.3
beta1, beta2, eps = 0.9, 0.999, 1e-8
m_x, m_y = 0.0, 0.0  # first moment
v_x, v_y = 0.0, 0.0  # second moment

trajectory_adam_manual = [(x.item(), y.item())]

for t in range(1, 51):
    loss = loss_fn(x, y)
    loss.backward()

    gx, gy = x.grad.item(), y.grad.item()

    m_x = beta1 * m_x + (1 - beta1) * gx
    m_y = beta1 * m_y + (1 - beta1) * gy
    v_x = beta2 * v_x + (1 - beta2) * gx**2
    v_y = beta2 * v_y + (1 - beta2) * gy**2

    m_hat_x = m_x / (1 - beta1**t)
    m_hat_y = m_y / (1 - beta1**t)
    v_hat_x = v_x / (1 - beta2**t)
    v_hat_y = v_y / (1 - beta2**t)

    with torch.no_grad():
        x -= lr * m_hat_x / (v_hat_x**0.5 + eps)
        y -= lr * m_hat_y / (v_hat_y**0.5 + eps)

    x.grad.zero_()
    y.grad.zero_()
    trajectory_adam_manual.append((x.item(), y.item()))

    if t <= 5 or t % 10 == 0:
        print(f"t={t:2d} | x={x.item():7.4f} y={y.item():7.4f} | "
              f"m=({m_hat_x:6.3f},{m_hat_y:6.3f}) v=({v_hat_x:6.3f},{v_hat_y:6.3f})")

print(f"\nFinal: ({x.item():.4f}, {y.item():.4f})")

## 5. AdamW vs Adam — Weight Decay

**Adam + L2 regularization** adds the penalty to the loss: `loss + lambda * ||w||^2`  
**AdamW (decoupled weight decay)** subtracts directly from weights: `w -= wd * w`

The difference matters because Adam's adaptive scaling interacts badly with L2 regularization — AdamW fixes this.

In [ ]:
model_adam = nn.Linear(20, 4)
model_adamw = nn.Linear(20, 4)
model_adamw.load_state_dict(model_adam.state_dict())

opt_adam = torch.optim.Adam(model_adam.parameters(), lr=0.01, weight_decay=0.1)
opt_adamw = torch.optim.AdamW(model_adamw.parameters(), lr=0.01, weight_decay=0.1)

torch.manual_seed(42)
weight_norms_adam, weight_norms_adamw = [], []

for step in range(200):
    x = torch.randn(32, 20)
    target = torch.randint(0, 4, (32,))

    for model, opt, norms in [(model_adam, opt_adam, weight_norms_adam),
                                (model_adamw, opt_adamw, weight_norms_adamw)]:
        opt.zero_grad()
        loss = nn.CrossEntropyLoss()(model(x), target)
        loss.backward()
        opt.step()
        norms.append(model.weight.data.norm().item())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weight_norms_adam, label="Adam + weight_decay (L2)", linewidth=2)
ax.plot(weight_norms_adamw, label="AdamW (decoupled WD)", linewidth=2)
ax.set_xlabel("Step")
ax.set_ylabel("Weight Norm")
ax.set_title("AdamW vs Adam: Effect on Weight Norms")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/adamw_vs_adam.png", dpi=100, bbox_inches="tight")
plt.show()
print("AdamW provides more consistent weight decay (the weights shrink more uniformly).")

## 6. Side-by-Side Optimizer Comparison

Let's compare SGD, SGD+Momentum, Adam, and AdamW on the same 2D loss surface:

In [ ]:
optimizers = {
    "SGD (lr=0.05)": lambda p, lr: torch.optim.SGD(p, lr=0.05),
    "SGD+Momentum": lambda p, lr: torch.optim.SGD(p, lr=0.05, momentum=0.9),
    "Adam (lr=0.3)": lambda p, lr: torch.optim.Adam(p, lr=0.3),
    "AdamW (lr=0.3)": lambda p, lr: torch.optim.AdamW(p, lr=0.3, weight_decay=0.01),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ["#e74c3c", "#2ecc71", "#3498db", "#9b59b6"]

for ax, (name, opt_fn), color in zip(axes.flat, optimizers.items(), colors):
    traj = optimize_trajectory(opt_fn, lr=0.05, steps=50)
    ax.contour(X, Y, Z, levels=np.logspace(-0.5, 3, 20), cmap="viridis", alpha=0.5)
    xs, ys = zip(*traj)
    ax.plot(xs, ys, ".-", color=color, linewidth=1.5, markersize=4)
    ax.plot(xs[0], ys[0], "go", markersize=10)
    ax.plot(1, 2, "r*", markersize=15)
    ax.set_title(f"{name}\nFinal: ({xs[-1]:.2f}, {ys[-1]:.2f})", fontsize=12)
    ax.set_xlabel("x"); ax.set_ylabel("y")

plt.suptitle("Optimizer Comparison on Elliptical Bowl", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/optimizer_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

### Convergence Speed Comparison

Let's plot how quickly each optimizer reduces the loss:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for (name, opt_fn), color in zip(optimizers.items(), colors):
    traj = optimize_trajectory(opt_fn, lr=0.05, steps=80)
    losses = [loss_fn(torch.tensor(x), torch.tensor(y)).item() for x, y in traj]
    ax.plot(losses, label=name, linewidth=2, color=color)

ax.set_xlabel("Step")
ax.set_ylabel("Loss (log scale)")
ax.set_yscale("log")
ax.set_title("Convergence Speed Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/convergence.png", dpi=100, bbox_inches="tight")
plt.show()
print("Adam-family optimizers typically converge faster on this problem.")

## 7. Learning Rate Schedulers Visualized

Schedulers adjust the learning rate during training. Let's visualize the most popular ones side by side.

In [ ]:
def get_lr_schedule(scheduler_cls, scheduler_kwargs, total_steps=1000, base_lr=0.1):
    """Record LR over steps for a given scheduler."""
    model = nn.Linear(10, 10)
    optimizer = torch.optim.SGD(model.parameters(), lr=base_lr)
    scheduler = scheduler_cls(optimizer, **scheduler_kwargs)
    lrs = []
    for _ in range(total_steps):
        lrs.append(optimizer.param_groups[0]["lr"])
        optimizer.step()
        scheduler.step()
    return lrs

total_steps = 1000
epochs = 100
steps_per_epoch = total_steps // epochs

schedules = {
    "StepLR": get_lr_schedule(
        torch.optim.lr_scheduler.StepLR,
        {"step_size": 30, "gamma": 0.5},
        total_steps, 0.1
    ),
    "CosineAnnealingLR": get_lr_schedule(
        torch.optim.lr_scheduler.CosineAnnealingLR,
        {"T_max": total_steps},
        total_steps, 0.1
    ),
    "OneCycleLR": get_lr_schedule(
        torch.optim.lr_scheduler.OneCycleLR,
        {"max_lr": 0.1, "total_steps": total_steps},
        total_steps, 0.01
    ),
    "ExponentialLR": get_lr_schedule(
        torch.optim.lr_scheduler.ExponentialLR,
        {"gamma": 0.995},
        total_steps, 0.1
    ),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]

for ax, (name, lrs), color in zip(axes.flat, schedules.items(), colors):
    ax.plot(lrs, color=color, linewidth=2)
    ax.set_title(name, fontsize=14)
    ax.set_xlabel("Step")
    ax.set_ylabel("Learning Rate")
    ax.grid(True, alpha=0.3)

plt.suptitle("Learning Rate Schedulers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/schedulers.png", dpi=100, bbox_inches="tight")
plt.show()

### Warmup + Cosine Decay (The Transformer Default)

Most transformer training uses a **linear warmup** followed by **cosine decay**. Let's build this with `SequentialLR` or a custom lambda:

In [ ]:
import math

def warmup_cosine_schedule(optimizer, warmup_steps, total_steps):
    """Linear warmup for warmup_steps, then cosine decay to 0."""
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return current_step / warmup_steps
        progress = (current_step - warmup_steps) / (total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

model = nn.Linear(10, 10)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scheduler = warmup_cosine_schedule(optimizer, warmup_steps=100, total_steps=1000)

lrs = []
for _ in range(1000):
    lrs.append(optimizer.param_groups[0]["lr"])
    optimizer.step()
    scheduler.step()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lrs, linewidth=2, color="#3498db")
ax.axvline(x=100, color="red", linestyle="--", alpha=0.7, label="End of warmup")
ax.set_xlabel("Step")
ax.set_ylabel("Learning Rate")
ax.set_title("Warmup + Cosine Decay (The Transformer Default)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/warmup_cosine.png", dpi=100, bbox_inches="tight")
plt.show()
print("Warmup prevents instability in early training when gradients are large.")

## 8. Parameter Groups

Different parts of a model may need different learning rates. For example, in transfer learning you want a smaller LR for the pretrained backbone and a larger LR for the new head.

In [ ]:
class TransferNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(20, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
        )
        self.head = nn.Linear(32, 4)

    def forward(self, x):
        return self.head(self.backbone(x))

model = TransferNet()

# Different LR for backbone vs head
optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": 1e-4, "name": "backbone"},
    {"params": model.head.parameters(), "lr": 1e-3, "name": "head"},
], weight_decay=0.01)

print("=== Parameter Groups ===")
for i, group in enumerate(optimizer.param_groups):
    n_params = sum(p.numel() for p in group["params"])
    print(f"  Group {i}: lr={group['lr']:.1e}, "
          f"weight_decay={group['weight_decay']}, "
          f"params={n_params:,}")

print("\nThis is standard for transfer learning:")
print("  - Backbone (pretrained): low LR (1e-4) to preserve learned features")
print("  - Head (random init): high LR (1e-3) to learn quickly")

# You can also add no_decay groups (e.g., skip weight decay for biases/norms)
decay_params = []
no_decay_params = []
for name, param in model.named_parameters():
    if "bias" in name or "norm" in name:
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer2 = torch.optim.AdamW([
    {"params": decay_params, "weight_decay": 0.01},
    {"params": no_decay_params, "weight_decay": 0.0},
], lr=1e-3)
print(f"\nDecay/no-decay split: {len(decay_params)} decayed, {len(no_decay_params)} not decayed")

## 9. Practical Optimizer Cheat Sheet

| Optimizer | Best for | Typical LR | Key features |
|-----------|---------|------------|-------------|
| SGD | CNNs, simple problems | 0.01-0.1 | Simple, good generalization |
| SGD+Momentum | CNNs, ResNets | 0.01-0.1 | Faster convergence |
| Adam | Transformers, GANs, fast prototyping | 1e-4-1e-3 | Adaptive, less tuning |
| AdamW | Transformers (standard) | 1e-4-1e-3 | Proper weight decay |

| Scheduler | Best for | Notes |
|-----------|---------|-------|
| StepLR | Quick experiments | Drop by gamma every N epochs |
| CosineAnnealing | General training | Smooth decay, good default |
| OneCycleLR | Fast training | Warmup + anneal, train fewer epochs |
| Warmup+Cosine | Transformers | Prevents early instability |

## 🧪 Exercise: Find the Best Optimizer and LR

**Task:** Train a small MLP on the synthetic dataset below. Try different optimizer/LR combinations and find the fastest converging setup.

Compare:
1. SGD with lr=0.01, 0.1, 0.5
2. Adam with lr=1e-4, 1e-3, 1e-2
3. Which converges fastest? Which diverges?

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
X = torch.randn(1000, 10)
W_true = torch.randn(10, 3)
y = (X @ W_true + 0.1 * torch.randn(1000, 3)).argmax(dim=1)
loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)

configs = [
    ("SGD lr=0.01", torch.optim.SGD, {"lr": 0.01}),
    ("SGD lr=0.1",  torch.optim.SGD, {"lr": 0.1}),
    ("SGD lr=0.5",  torch.optim.SGD, {"lr": 0.5}),
    ("Adam lr=1e-4", torch.optim.Adam, {"lr": 1e-4}),
    ("Adam lr=1e-3", torch.optim.Adam, {"lr": 1e-3}),
    ("Adam lr=1e-2", torch.optim.Adam, {"lr": 1e-2}),
]

fig, ax = plt.subplots(figsize=(10, 5))

for name, opt_cls, opt_kwargs in configs:
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
    optimizer = opt_cls(model.parameters(), **opt_kwargs)
    criterion = nn.CrossEntropyLoss()

    losses = []
    for epoch in range(30):
        epoch_loss = 0
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))

    ax.plot(losses, label=name, linewidth=2)

ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Optimizer & LR Comparison")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 2.5)
plt.tight_layout()
plt.savefig("/tmp/pytorch-reference-guide/notebooks/exercise_optim.png", dpi=100, bbox_inches="tight")
plt.show()
print("Which optimizer/LR combo converges fastest? Try adding your own!")

### Try It Yourself

1. Add `AdamW` with `weight_decay=0.01` to the comparison
2. Try `OneCycleLR` with `Adam` — does it help?
3. What happens with very large LR (e.g., `lr=10.0` for SGD)?

In [ ]:
# Try it yourself! Add your own optimizer experiments here:
# Example: try AdamW
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(30):
    epoch_loss = 0
    for bx, by in loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(loader))

print(f"AdamW final loss: {losses[-1]:.4f}")

## Key Takeaways

1. **SGD** is simple and generalizes well, but needs careful LR tuning and benefits from momentum
2. **SGD + Momentum (0.9)** reduces oscillation and speeds convergence — always use momentum with SGD
3. **Adam** adapts per-parameter learning rates using running mean/variance of gradients
4. **AdamW** decouples weight decay from the gradient update — use this instead of Adam when using weight decay
5. **Learning rate is the most important hyperparameter** — always tune it first
6. **Warmup + Cosine Decay** is the standard schedule for transformers
7. **OneCycleLR** enables fast training with super-convergence
8. **Parameter groups** let you set different LR/weight decay for different parts of the model
9. **No weight decay on biases and norms** — a standard best practice

**Congratulations!** You've completed the PyTorch Learning Guide. You now have a solid foundation in tensors, autograd, neural networks, training, and optimization.